# Frost-Days — Statistiques descriptives & contrôle qualité

Ce notebook vérifie la qualité des données Météo-France et la cohérence des résultats :

1. Unité de la colonne `TN` (°C décimaux vs autre).
2. Taux de valeurs manquantes par station.
3. Distances stations / commune.
4. Cohérence des jours de gel.
5. Comparaison aux exports partiels de vérification (à fournir).

Exécuter depuis la racine du projet : `uv run jupyter lab notebooks/exploration.ipynb`.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))  # racine du projet

import pandas as pd
import matplotlib.pyplot as plt

from frost_days import config
from frost_days.weather import load_department_tn
from frost_days.stations import list_stations, rank_by_distance
from frost_days.communes import get_commune_coords
from frost_days.frost import compute_stats, missing_ratio

DEPT = "75"
START = pd.Timestamp(config.PERIOD_START)
END = pd.Timestamp(config.PERIOD_END)

: 

## 1. Unité de TN
On vérifie l'amplitude des valeurs : des °C décimaux donnent typiquement une plage [-30, +30].

In [ ]:
df = load_department_tn(DEPT)
print(df.shape)
df[config.COL_TN].describe()

## 2. Taux de valeurs manquantes par station (sur la période cible)

In [ ]:
period = df[(df[config.COL_DATE] >= START) & (df[config.COL_DATE] <= END)]
expected = (END.normalize() - START.normalize()).days + 1
miss = (
    period.groupby(config.COL_STATION)[config.COL_TN]
    .apply(lambda s: 1 - s.notna().sum() / expected)
    .sort_values()
)
print(f"Stations sous le seuil de {config.MAX_MISSING_RATIO:.0%} : ", (miss <= config.MAX_MISSING_RATIO).sum(), "/", len(miss))
miss.head(15)

## 3. Stations les plus proches d'une commune

In [ ]:
lat, lon = get_commune_coords("Paris", DEPT)
ranked = rank_by_distance(list_stations(DEPT), lat, lon)
ranked.head(10)

## 4. Cohérence des jours de gel

In [ ]:
stats = compute_stats("Paris", DEPT, START, END)
print("Station :", stats.station_name, f"({stats.distance_km:.1f} km, {stats.missing_ratio:.0%} manquants)")
print("Total :", stats.total_frost_days, "| Moyenne/an :", round(stats.avg_frost_days_per_year, 1))
stats.frost_days_per_year.plot(kind="bar", title="Jours de gel par année — Paris")
plt.tight_layout(); plt.show()

In [ ]:
# Saisonnalité : fréquence de gel par jour de l'année
pdoy = stats.per_day_of_year.copy()
pdoy["date"] = pd.to_datetime("2001-" + pdoy.index, format="%Y-%m-%d")
pdoy.sort_values("date").plot(x="date", y="freq_relative", legend=False, figsize=(11, 3), title="Fréquence de gel par jour de l'année")
plt.ylabel("Fréquence"); plt.tight_layout(); plt.show()

## 5. Comparaison aux exports de vérification

Quand l'enseignant fournit les exports partiels (période 2013-2023), les charger ici et comparer
commune par commune le `total_frost_days` calculé à la valeur attendue.

In [ ]:
# Exemple de structure de validation (à adapter au format réel des exports) :
#
# expected = pd.read_csv("../data/verification/exports.csv")
# for _, row in expected.iterrows():
#     s = compute_stats(row.commune, row.departement, row.debut, row.fin)
#     print(row.commune, s.total_frost_days, "vs", row.total_attendu)
pass

## Carte interactive

Cette cellule affiche une carte Folium centrée sur la France.


In [ ]:
import sys, subprocess

try:

    import folium

except ImportError:

    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'folium'])

    import folium



# Carte centrée sur la France

m = folium.Map(location=[46.5, 2.0], zoom_start=6)

# Exemple de marqueur

folium.Marker([48.8566, 2.3522], popup='Paris').add_to(m)

m